# 01 · Extracción de texto (PDF → texto plano)

<a href="https://colab.research.google.com/github/manuelarguelles/tyv-demo-colab/blob/main/notebooks/01_extraccion_texto.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>

Primer paso del pipeline de filtrado curricular de **Terry & Valdez**: antes de
que cualquier modelo de lenguaje vea un currículum, el sistema necesita
convertirlo de PDF/DOCX a **texto plano**. Este notebook construye un PDF de
ejemplo (100% ficticio) y lo procesa con la misma herramienta que usa el
sistema real: **Poppler** (`pdftotext`), un extractor de código abierto.

**Por qué no usar una librería Python "pura" (como `PyPDF2`) en su lugar:**
`pdftotext -layout` conserva el orden espacial del texto en la página — dos
columnas, tablas, secciones alineadas — que es exactamente lo que hace un CV
legible tanto para una persona como para un modelo de lenguaje. Extraer sin
`-layout` puede entremezclar columnas y romper el orden de lectura.


## 1. Instalar Poppler (el extractor real)

En Colab, Poppler se instala vía `apt-get`; es la misma versión que correría en un servidor Linux.

In [ ]:
!apt-get -qq update && apt-get -qq install -y poppler-utils > /dev/null
!pdftotext -v


## 2. Construir un CV de ejemplo (ficticio) como PDF

Usamos `reportlab` para generar un PDF real desde texto plano — así el notebook es 100% autocontenido y no depende de subir un archivo.

In [ ]:
!pip install -q reportlab

CV_TEXTO = """ALEJANDRA ROJAS MEDINA
Lima, Perú · a.rojas.medina@ejemplo.com · +51 987 654 321
Jr. Los Alamos 245, San Isidro, Lima

FORMACIÓN ACADÉMICA
Bachiller en Derecho — Universidad Nacional Mayor de San Marcos (2015 – 2020)
Diplomado en Derecho Laboral — Pontificia Universidad Católica del Perú (2021)
Certificación en Protección de Datos Personales — Indecopi (2022)

EXPERIENCIA PROFESIONAL
Asistente Legal Junior — Estudio Fernández & Asociados (2020 – 2022)
  Apoyo en la elaboración de contratos laborales y absolución de consultas
  de clientes corporativos sobre normativa de protección de datos.

Analista Legal — Grupo Andino S.A.C. (2022 – Presente)
  Responsable de la revisión de políticas internas de privacidad y de la
  coordinación con el área de Recursos Humanos en procesos disciplinarios.

CONOCIMIENTOS TÉCNICOS
Manejo de bases de datos jurisprudenciales (LP, Actualidad Jurídica).
Redacción de informes legales y absolución de consultas escritas.
Nivel intermedio de inglés (certificado ICPNA).

Fecha de nacimiento: 14 de marzo de 1994
DNI: 45678912"""

from reportlab.lib.pagesizes import LETTER
from reportlab.pdfgen import canvas
from reportlab.lib.units import cm

def construir_pdf_desde_texto(texto: str, ruta_salida: str) -> None:
    """Genera un PDF simple, una línea por línea de texto — suficiente para
    demostrar extracción; el sistema real recibe el PDF que sube el candidato."""
    c = canvas.Canvas(ruta_salida, pagesize=LETTER)
    ancho, alto = LETTER
    y = alto - 2 * cm
    c.setFont("Helvetica", 10)
    for linea in texto.splitlines():
        if y < 2 * cm:
            c.showPage()
            c.setFont("Helvetica", 10)
            y = alto - 2 * cm
        c.drawString(2 * cm, y, linea)
        y -= 0.45 * cm
    c.save()

construir_pdf_desde_texto(CV_TEXTO, "cv_ejemplo.pdf")
print("PDF de ejemplo creado: cv_ejemplo.pdf")


## 3. Extraer el texto con `pdftotext -layout`

Esta es la llamada real que hace el sistema (vía `subprocess`), aplicada aquí sobre nuestro PDF de ejemplo.

In [ ]:
import subprocess

def extraer_texto_pdf(ruta_pdf: str) -> str:
    """Equivalente a la función `extraer()` del servidor real:
    `pdftotext -layout <pdf> -` envía el resultado a stdout."""
    resultado = subprocess.run(
        ["pdftotext", "-layout", ruta_pdf, "-"],
        capture_output=True, text=True, check=True,
    )
    return resultado.stdout

texto_extraido = extraer_texto_pdf("cv_ejemplo.pdf")
print(texto_extraido)


## 4. Por qué `-layout` importa (comparación)

Repetimos la extracción sin `-layout` para ver la diferencia. Con un CV de
una sola columna como el de ejemplo el efecto es sutil, pero en un CV real
con tablas o dos columnas (fechas a la derecha, cargos a la izquierda) la
diferencia es la que separa un texto legible de un texto con las palabras
mezcladas fuera de orden.

In [ ]:
resultado_sin_layout = subprocess.run(
    ["pdftotext", "cv_ejemplo.pdf", "-"],
    capture_output=True, text=True, check=True,
).stdout

print("── CON -layout (primeras 5 líneas) ──")
print("\n".join(texto_extraido.splitlines()[:5]))
print()
print("── SIN -layout (primeras 5 líneas) ──")
print("\n".join(resultado_sin_layout.splitlines()[:5]))


## 5. Límite de tamaño

El sistema real recorta el CV a un máximo de caracteres antes de enviarlo al
modelo (calibrado sobre 287 CVs reales del proyecto: percentil 90 ≈ 13 600
caracteres, ~3 páginas). Esto evita enviar certificados adjuntos completos
como si fueran parte del currículum.

In [ ]:
MAX_CV = 12_000  # caracteres — mismo límite que usa el sistema real

def recortar_a_limite(texto: str, limite: int = MAX_CV) -> str:
    if len(texto) <= limite:
        return texto
    return texto[:limite] + "\n[... recortado: documento más largo que el límite operativo ...]"

texto_final = recortar_a_limite(texto_extraido)
print(f"Longitud del CV extraído: {len(texto_extraido)} caracteres (límite: {MAX_CV})")


## Siguiente paso

El texto extraído (`texto_final`) todavía contiene datos personales —
nombre, correo, teléfono, DNI, fecha de nacimiento, dirección. Antes de que
cualquier modelo de IA lo vea, pasa por la etapa de **anonimización**
→ ver `02_anonimizacion.ipynb`.

---
*Este material es contenido educativo de apoyo a una tesis de maestría (Terry & Valdez — sistema de filtrado curricular). El CV usado es 100% ficticio, construido para esta demostración. Ningún dato de candidatos reales del proyecto se publica en este repositorio: ver `materiales/README.md` para trabajar con datos reales de forma local.*
